© 2026 by Tamás Takács is licensed under CC BY-NC-SA 4.0. To view a copy of this license, visit https://creativecommons.org/licenses/by-nc-sa/4.0/

English translation managed by Tamás Takács. The translation was produced with AI assistance.

# **Epic Emojis**

<div style="font-size: 14px; color: #6e8192; line-height: 1.5;">
  <div style="display: flex; align-items: center; gap: 5px; margin-bottom: 5px;">
    <span style="font-size: 18px; color: #6e8192;">🎯</span>
    <span>AI National Student Olympiad Qualifier</span>
  </div>
  <div style="display: flex; align-items: center; gap: 5px;">
    <span style="font-size: 18px; color: #6e8192;">🧠</span>
    <span>Natural Language Processing</span>
  </div>
  <div style="display: flex; align-items: center; gap: 5px;">
    <span style="font-size: 18px; color: #6e8192;">🏆</span>
    <span>100 points</span>
  </div>
  <div style="display: flex; align-items: center; gap: 5px;">
    <span style="font-size: 18px; color: #6e8192;">🗓️</span>
    <span>24 May 2025</span>
  </div>
</div>

**Name:** [WRITE YOUR NAME HERE]

**Contestant ID:** [WRITE YOUR CONTESTANT ID HERE]

<img src="https://drive.google.com/uc?export=view&id=1vhGZdZ95KsizBHg8C9UifQkOefDkkF8K" alt="petike" style="width:150px;">

**Emoji Emma** is a self-proclaimed film aesthete who is obsessed with the world of cinema and with emojis. She has devoted her life to summarizing every existing film title with a single well-chosen **emoji sequence** (4-5 emojis).  
😅 “If a film can't be described with three emojis, it doesn't even deserve to be watched!”.

In this task you have to train a language model that is **able to generate emoji sequences from English film titles**. For this you get an existing dataset as well as a pre-trained model (EmojiLM), which you need to fine-tune further and then evaluate.

### **Steps of the task**

1. **Data loading and exploration** (10 points)  

2. **Data cleaning** (5 points)  

3. **Loading the model** (5 points)  

4. **Data preprocessing** (20 points)  

5. **Baseline evaluation, non-fine-tuned model** (10 points)  

6. **Training the model** (25 points)  

7. **Evaluating the model and visualization** (3 points)  

8. **Visualization of the results** (7 points)

9. **Visualization of the Attention Mechanism** (15 points)

During the solution you will use the tools of NLP and generative language modelling to express the content hidden behind the titles with emojis. The ultimate goal is for the model to be able to **create witty emoji montages for new film titles as well**, exactly the way Emoji Emma dreamed it up.

## **Useful links**

- [Hugging Face Transformers - Documentation](https://huggingface.co/docs/transformers/index)  

- [Hugging Face Datasets - Documentation](https://huggingface.co/docs/datasets/index)  

- [T5 model - principles](https://arxiv.org/abs/1910.10683)  

- [BLEU score - interpretation and example](https://machinelearningmastery.com/calculate-bleu-score-for-text-python/)  

- [Python Regex - Filtering emojis](https://www.regular-expressions.info/unicode.html)


## **Required Imports**

In [ ]:
!pip install --q evaluate nltk
!pip install --q datasets

import torch
import re
import os
import gc
import matplotlib.pyplot as plt
import time
import nltk
from torch.utils.data import DataLoader
from transformers import get_scheduler, DataCollatorForSeq2Seq
from torch.optim import AdamW
from datasets import Dataset
from tqdm import tqdm
from evaluate import load
from nltk.tokenize import word_tokenize

nltk.download('punkt_tab')
bleu = load("bleu")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.4 MB/s eta 0:00:00


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## **Downloading the Dataset**

The downloaded `.csv` file contains the following two columns:

- **title**: an English film title (e.g. `"Finding Nemo"`, `"The Matrix"`)
- **emojis**: the emoji sequence belonging to the given film title, which tries to convey the content, mood or characters of the film (e.g. `"🐠🔍🌊"` or `"👓💊🤖"`)

You will use this data to train the language model: the goal is for the model to learn to suggest similarly witty emojis for new film titles.

In [ ]:
!gdown 19d_GJ-DLoTiQegV-oPNIIuDlk_UAcuRt

Downloading...
From: https://drive.google.com/uc?id=19d_GJ-DLoTiQegV-oPNIIuDlk_UAcuRt
To: /content/emoji_dataset.csv
100% 41.2k/41.2k [00:00<00:00, 107MB/s]


### **Loading the Dataset**

In [ ]:
import pandas as pd

dataset = pd.read_csv("emoji_dataset.csv")

In [ ]:
dataset.head()

,title,emojis
0,The Shawshank Redemption,NaN
1,The Godfather,👨‍👦🕴️🤵🔫🇮🇹
2,The Dark Knight,🦇🤡🃏🚓🚨
3,The Godfather Part II,🧓👶🔫💼🇮🇹
4,12 Angry Men,👨‍⚖️🔍🤔🔒👥


# **Task 1 (10 points)**

Let's select **10 random examples** from the dataset and examine what the data looks like.

With this step we can gain insight into what kind of film titles and corresponding emojis appear in the training set. It is worth observing how clear the connection is between the title and the emoji sequence, for example whether it refers back to the genre, the characters or the events.

Let's examine how many different film titles and how many unique emojis appear in the dataset.

- For the film titles, count the unique titles after filtering out the duplicate rows.
- For the emojis, however, it is not the whole emoji sequences that count, but all the unique emoji characters that occur, so each individual emoji is a separate unit.

In [ ]:
# --------- YOUR OWN SOLUTION GOES HERE --------- (~14-20 lines)

# ----------- END OF YOUR SOLUTION -------------

# **Task 2 (5 points)**

First let's clean our data:

- Delete the rows where the film title or the emojis are missing.
- Remove any duplicated rows so that the model does not learn the same information multiple times.

Then select 60 random examples from the dataset cleaned this way, with which we will train the model.

In [ ]:
# --------- YOUR OWN SOLUTION GOES HERE --------- (~3-5 lines)

# ----------- END OF YOUR SOLUTION -------------

<details>
<summary><strong>💥 If Colab warns you that the GPU memory is full, click here and run this cell!</strong></summary>

This helper function frees up the GPU, deletes the unnecessary variables, empties the cache, and tries to return the memory to the system.  
❗ **Only use it if the system warns you or an error message appears due to memory usage!**

```python
def clear_memory():
    if "inputs" in globals():
        del globals()["inputs"]
    if "model" in globals():
        del globals()["model"]
    if "processor" in globals():
        del globals()["processor"]
    if "trainer" in globals():
        del globals()["trainer"]
    if "peft_model" in globals():
        del globals()["peft_model"]
    if "bnb_config" in globals():
        del globals()["bnb_config"]
    time.sleep(2)
    gc.collect()
    time.sleep(2)
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    time.sleep(2)
    gc.collect()
    time.sleep(2)

    print(f"GPU allocated memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU reserved memory: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

clear_memory()
```
</details>

# **Task 3 (5 points)**

Load the given pre-trained language model from the Hugging Face model hub:  
🔗 [`KomeijiForce/t5-base-emojilm`](https://huggingface.co/KomeijiForce/t5-base-emojilm)

This is a T5-based model that was trained specifically for translating text into emojis. The model expects the input to start with a *translate into emojis:* prefix, followed by the film title.

For loading it you will need the following components:
- **Tokenizer**: processes the input for the model,
- **Generative model**: the trained T5 model itself, which produces the emojis.

In [ ]:
# --------- YOUR OWN SOLUTION GOES HERE --------- (~4-5 lines)

# ----------- END OF YOUR SOLUTION -------------

# **Task 4 (20 points)**

To train the model, the data must be brought into an appropriate format. You can prepare the dataset in two ways:

- Create a `Dataset` object (e.g. with the Hugging Face `datasets` library), then write your own tokenizing function that prepares the inputs and labels.
- OR create a `DataLoader`, for which you can use a `collate_fn` and helper functions.

**Important technical rules:**
- The inputs must be padded to a uniform length so that they can be processed in batches.
- In the case of the output labels, the padding tokens must be set to the value `-100` so that the loss function ignores them.

In [ ]:
# --------- YOUR OWN SOLUTION GOES HERE --------- (~12-20 lines)

# ----------- END OF YOUR SOLUTION -------------

# **Task 5 (10 points)**

Before we train anything, let's assess **what the default, non-fine-tuned model can do**. For this you need to prepare the following:

Write two helper functions:
- The first passes one film title at a time to the model and returns the emoji sequence generated for it.
- The second goes through the entire training set, and by comparing the real and generated emoji sequences it computes the BLEU score.

**BLEU** is an automatic evaluation metric that is often used to measure the performance of language models; the higher it is, the better the match of the generated text (in this case emoji sequence) with the expected one.


In [ ]:
# --------- YOUR OWN SOLUTION GOES HERE --------- (~18-25 lines)

# ----------- END OF YOUR SOLUTION -------------

# **Task 6 (25 points)**

In this task the goal is to fine-tune the EmojiLM model on the selected sample, either with your own code or with the help of the Hugging Face `Trainer` class.

During training you have to apply the so-called **early stopping** technique, which interrupts learning if it no longer improves.  

Since we are working with a small dataset, there is no validation set now, so the early stopping must be based on the **loss value of the training set**.

**Two possible approaches:**
- Write the training loop by hand (`train loop`), where you set up yourself:
  - the optimizer,
  - the learning rate,
  - the scheduler,
  - the tracking of the `loss` per epoch.
- You can use the Hugging Face `Trainer` class if you configure it appropriately (e.g. `EarlyStoppingCallback`, `TrainingArguments`).

**Saving the loss values:**  
During learning, **save the current train loss value** at the end of every epoch so that the learning curve can be visualized later.

---

### **Scoring**

The achievable score depends on the training performance (loss) of the model:

| Train loss value achieved | Achievable points |
|------------------------|---------------------|
| < 1                    | 25 points           |
| < 2                    | 20 points           |
| < 4                    | 15 points           |
| < 5                    | 10 points           |

In [ ]:
# --------- YOUR OWN SOLUTION GOES HERE --------- (~25-35 lines)

# ----------- END OF YOUR SOLUTION -------------

# **Task 7 (3 points)**

Now that you have trained the model, let's see how it performs on concrete examples!

Run the model on the first 10 film titles of the dataset, and print:

- the original film title,
- the emoji sequence generated by the model,
- as well as the real (expected) emojis belonging to the given example.

In [ ]:
# --------- YOUR OWN SOLUTION GOES HERE --------- (~5-6 lines)

# ----------- END OF YOUR SOLUTION -------------

# **Task 8 (7 points)**

Now let's evaluate how much the model improved during training.

Your tasks:

1. Run the trained model on the first 10 film titles (just like before).
   - Compare the generated emojis with the real ones.

2. Create a figure that shows the `train loss` values recorded during training per epoch.

3. Compute the BLEU score for the trained model as well.
   - Compare the value with the BLEU score of the previously measured (untrained) model.

In [ ]:
# --------- YOUR OWN SOLUTION GOES HERE --------- (~8-15 lines)

# ----------- END OF YOUR SOLUTION -------------

# **Task 9 (15 points)**

After you have trained the T5-based model on the given title-emoji pair data, visualize how the attention mechanism works. For this, select a few examples from the test set and create a visualization that shows which input title words the generated emojis attend to the most in the decoder attention layers.

Use a heatmap display to depict the decoder-encoder attention weights (cross-attention).

Requirements:
- Select at least 5 examples from the test set.
- Use attention weights (`cross_attentions`) from the T5 model.
- Create at least 5 heatmaps (e.g. with the help of `matplotlib` or `seaborn`).

In [ ]:
# --------- YOUR OWN SOLUTION GOES HERE --------- (~15-25 lines)

# ----------- END OF YOUR SOLUTION -------------

---

## 🎉 Congratulations!

You have reached the end of the exercise set - excellent work!  

---